# Notebook 2 — Train ONE ResU-Net Model on GPU

Run this notebook only after Notebook 1 has completed and you have **restarted the kernel**.

This notebook:

- imports **PyTorch only** for GPU work
- never imports CUDA-Q
- trains **one experiment and one seed**
- saves the result to disk
- uses batch size 2, base filters 16, gradient accumulation 4, workers 0
- does not keep giant per-pixel probability arrays in RAM

After one experiment finishes, **restart the kernel**, change the experiment/seed below, and run Notebook 2 again.

In [ ]:
# ============================================================
# STEP 1 — CHANGE ONLY EXPERIMENT AND SEED
# ============================================================

EXPERIMENT = "ALL_14"

# Allowed:
# "ALL_14"
# "MI_8"
# "mRMR_8"
# "QAOA_6"
# "QAOA_8"
# "QAOA_10"

SEED = 42

# Recommended seeds for the final manuscript:
# 42, 52, 62

# Safe settings — do not increase initially.
BATCH_SIZE = 2
BASE_FILTERS = 16
GRAD_ACCUM_STEPS = 4
MAX_EPOCHS = 60
PATIENCE = 10

print("EXPERIMENT:", EXPERIMENT)
print("SEED      :", SEED)

## Define the single-model PyTorch workflow

    This cell defines the dataset, ResU-Net, Focal–Dice loss, training, validation threshold selection, and holdout evaluation.

In [ ]:
#!/usr/bin/env python3
"""
Stage 2 — Train ONE ResU-Net experiment/seed on the NVIDIA GPU.

IMPORTANT:
- This script imports PyTorch but DOES NOT import CUDA-Q.
- Every invocation is a fresh Linux process.
- When the script exits, all GPU memory is released by the OS.

Example:
python 02_train_one_model.py --experiment QAOA_8 --seed 42
"""

from pathlib import Path
import argparse
import gc
import json
import math
import os
import random
import time

# Set allocator before importing torch.
os.environ.setdefault(
    "PYTORCH_CUDA_ALLOC_CONF",
    "expandable_segments:True,max_split_size_mb:128"
)

import h5py
import numpy as np
import pandas as pd
from scipy.ndimage import binary_dilation, binary_erosion

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DATA_ROOT_DEFAULT = Path(r"/home/auhspark2/dhiraj/landslide")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class LandslideDataset(Dataset):
    def __init__(self, records, channels, mean, std, augment=False):
        self.records = records.reset_index(drop=True)
        self.channels = list(channels)
        self.mean = mean[self.channels].astype(np.float32)
        self.std = std[self.channels].astype(np.float32)
        self.augment = augment

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        with h5py.File(row.image_path, "r") as f:
            x = np.asarray(f["img"][...], dtype=np.float32)[..., self.channels]
        with h5py.File(row.mask_path, "r") as f:
            y = np.asarray(f["mask"][...], dtype=np.float32)

        for j in range(x.shape[-1]):
            bad = ~np.isfinite(x[...,j])
            if bad.any():
                x[...,j][bad] = self.mean[j]

        if self.augment:
            if random.random() < 0.5:
                x = np.flip(x, axis=1); y = np.flip(y, axis=1)
            if random.random() < 0.5:
                x = np.flip(x, axis=0); y = np.flip(y, axis=0)
            k = random.randint(0,3)
            if k:
                x = np.rot90(x, k, axes=(0,1))
                y = np.rot90(y, k, axes=(0,1))

        x = np.ascontiguousarray((x-self.mean[None,None,:]) / self.std[None,None,:])
        y = np.ascontiguousarray(y)

        x = torch.from_numpy(x).permute(2,0,1).float()
        y = torch.from_numpy(y)[None,...].float()
        return x, y, int(row.id)

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,out_ch,3,padding=1,bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch,out_ch,3,padding=1,bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.dropout = nn.Dropout2d(dropout) if dropout else nn.Identity()
        self.skip = nn.Conv2d(in_ch,out_ch,1,bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self,x):
        identity = self.skip(x)
        x = F.relu(self.bn1(self.conv1(x)), inplace=True)
        x = self.dropout(x)
        x = self.bn2(self.conv2(x))
        return F.relu(x + identity, inplace=True)

class UpBlock(nn.Module):
    def __init__(self,in_ch,skip_ch,out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch,out_ch,2,stride=2)
        self.block = ResidualBlock(out_ch+skip_ch,out_ch)
    def forward(self,x,skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x,size=skip.shape[-2:],mode="bilinear",align_corners=False)
        return self.block(torch.cat([x,skip],dim=1))

class ResUNet(nn.Module):
    def __init__(self,in_channels,base=16):
        super().__init__()
        self.enc1=ResidualBlock(in_channels,base)
        self.enc2=ResidualBlock(base,base*2)
        self.enc3=ResidualBlock(base*2,base*4)
        self.enc4=ResidualBlock(base*4,base*8)
        self.pool=nn.MaxPool2d(2)
        self.bottleneck=ResidualBlock(base*8,base*16,0.2)
        self.up4=UpBlock(base*16,base*8,base*8)
        self.up3=UpBlock(base*8,base*4,base*4)
        self.up2=UpBlock(base*4,base*2,base*2)
        self.up1=UpBlock(base*2,base,base)
        self.out=nn.Conv2d(base,1,1)
    def forward(self,x):
        e1=self.enc1(x)
        e2=self.enc2(self.pool(e1))
        e3=self.enc3(self.pool(e2))
        e4=self.enc4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        d4=self.up4(b,e4)
        d3=self.up3(d4,e3)
        d2=self.up2(d3,e2)
        d1=self.up1(d2,e1)
        return self.out(d1)

class FocalDiceLoss(nn.Module):
    def __init__(self,alpha=0.75,gamma=2.0):
        super().__init__()
        self.alpha=alpha; self.gamma=gamma
    def forward(self,logits,targets):
        bce=F.binary_cross_entropy_with_logits(logits,targets,reduction="none")
        prob=torch.sigmoid(logits)
        pt=prob*targets+(1-prob)*(1-targets)
        at=self.alpha*targets+(1-self.alpha)*(1-targets)
        focal=(at*((1-pt)**self.gamma)*bce).mean()
        inter=(prob*targets).sum((0,2,3))
        denom=prob.sum((0,2,3))+targets.sum((0,2,3))
        dice=1-((2*inter+1e-6)/(denom+1e-6)).mean()
        return 0.5*focal+0.5*dice

def autocast_ctx():
    return torch.amp.autocast("cuda", enabled=True)

def scaler_obj():
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=True)

def boundary_f1(gt,pred,tol=2):
    gt=gt.astype(bool); pred=pred.astype(bool)
    gb=gt ^ binary_erosion(gt)
    pb=pred ^ binary_erosion(pred)
    if gb.sum()==0:
        return np.nan
    gd=binary_dilation(gb,iterations=tol)
    pdil=binary_dilation(pb,iterations=tol)
    p=np.logical_and(pb,gd).sum()/(pb.sum()+1e-12)
    r=np.logical_and(gb,pdil).sum()/(gb.sum()+1e-12)
    return float(2*p*r/(p+r+1e-12))

def update_hist(pos_hist,neg_hist,prob,truth,bins):
    idx=np.minimum((prob*(bins-1)).astype(np.int32),bins-1)
    flat_idx=idx.reshape(-1)
    flat_y=truth.reshape(-1).astype(np.uint8)
    pos=np.bincount(flat_idx[flat_y==1],minlength=bins)
    neg=np.bincount(flat_idx[flat_y==0],minlength=bins)
    pos_hist += pos
    neg_hist += neg

def hist_curves(pos_hist,neg_hist):
    # Threshold sweeps high -> low.
    tp=np.cumsum(pos_hist[::-1]).astype(np.float64)
    fp=np.cumsum(neg_hist[::-1]).astype(np.float64)
    P=max(float(pos_hist.sum()),1.0)
    N=max(float(neg_hist.sum()),1.0)
    recall=tp/P
    precision=tp/(tp+fp+1e-12)
    fpr=fp/N

    # Add natural starting points.
    recall2=np.concatenate([[0.0],recall])
    precision2=np.concatenate([[1.0],precision])
    fpr2=np.concatenate([[0.0],fpr])

    # Approximate trapezoidal areas from all pixels summarized in 4096 bins.
    pr_auc=float(np.trapezoid(precision2,recall2))
    roc_auc=float(np.trapezoid(recall2,fpr2))
    return precision,recall,fpr,pr_auc,roc_auc

def best_threshold_from_hist(pos_hist,neg_hist):
    tp=np.cumsum(pos_hist[::-1]).astype(np.float64)
    fp=np.cumsum(neg_hist[::-1]).astype(np.float64)
    total_pos=float(pos_hist.sum())
    fn=total_pos-tp
    f1=2*tp/(2*tp+fp+fn+1e-12)
    best=int(np.argmax(f1))
    bins=len(pos_hist)
    # Reverse index maps to probability bin.
    bin_index=(bins-1)-best
    threshold=bin_index/(bins-1)
    return float(threshold), float(f1[best])

@torch.no_grad()
def validation_hist(model,loader,device,bins=4096):
    model.eval()
    ph=np.zeros(bins,np.int64); nh=np.zeros(bins,np.int64)
    for x,y,_ in loader:
        x=x.to(device,non_blocking=False)
        prob=torch.sigmoid(model(x)).squeeze(1).cpu().numpy()
        truth=y.squeeze(1).numpy().astype(np.uint8)
        update_hist(ph,nh,prob,truth,bins)
        del x,prob,truth
    return ph,nh

@torch.no_grad()
def test_metrics(model,loader,device,threshold,out_prefix,bins=4096,save_arrays=False):
    model.eval()
    ph=np.zeros(bins,np.int64); nh=np.zeros(bins,np.int64)
    TP=TN=FP=FN=0
    patch_rows=[]
    bf=[]
    saved_probs=[] if save_arrays else None
    saved_pred=[] if save_arrays else None
    saved_true=[] if save_arrays else None
    saved_ids=[] if save_arrays else None

    start=time.perf_counter()
    for x,y,ids in loader:
        x=x.to(device,non_blocking=False)
        prob=torch.sigmoid(model(x)).squeeze(1).cpu().numpy().astype(np.float32)
        truth=y.squeeze(1).numpy().astype(np.uint8)
        pred=(prob>=threshold).astype(np.uint8)
        update_hist(ph,nh,prob,truth,bins)

        for i in range(len(ids)):
            gt=truth[i].astype(bool); pr=pred[i].astype(bool)
            tp=int(np.logical_and(gt,pr).sum())
            tn=int(np.logical_and(~gt,~pr).sum())
            fp=int(np.logical_and(~gt,pr).sum())
            fn=int(np.logical_and(gt,~pr).sum())
            TP+=tp; TN+=tn; FP+=fp; FN+=fn
            b=boundary_f1(gt,pr,2)
            if np.isfinite(b): bf.append(b)
            patch_rows.append({
                "id":int(ids[i]),"tp":tp,"tn":tn,"fp":fp,"fn":fn,
                "patch_F1":2*tp/(2*tp+fp+fn+1e-12),
                "patch_IoU":tp/(tp+fp+fn+1e-12)
            })
        if save_arrays:
            saved_probs.append(prob.astype(np.float16))
            saved_pred.append(pred)
            saved_true.append(truth)
            saved_ids.extend([int(v) for v in ids])

        del x,prob,truth,pred

    torch.cuda.synchronize()
    sec=time.perf_counter()-start

    precision=TP/(TP+FP+1e-12)
    recall=TP/(TP+FN+1e-12)
    specificity=TN/(TN+FP+1e-12)
    f1=2*TP/(2*TP+FP+FN+1e-12)
    iou=TP/(TP+FP+FN+1e-12)
    bal=0.5*(recall+specificity)
    acc=(TP+TN)/(TP+TN+FP+FN+1e-12)
    # MCC
    denom=math.sqrt((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN)+1e-12)
    mcc=(TP*TN-FP*FN)/denom
    # Kappa
    total=TP+TN+FP+FN
    po=(TP+TN)/(total+1e-12)
    pe=((TP+FP)*(TP+FN)+(FN+TN)*(FP+TN))/(total*total+1e-12)
    kappa=(po-pe)/(1-pe+1e-12)

    pr,rc,fpr,pr_auc,roc_auc=hist_curves(ph,nh)

    if save_arrays:
        np.savez_compressed(
            str(out_prefix)+"_primary_predictions.npz",
            probabilities=np.concatenate(saved_probs),
            predictions=np.concatenate(saved_pred),
            truth=np.concatenate(saved_true),
            ids=np.asarray(saved_ids)
        )

    np.savez_compressed(
        str(out_prefix)+"_curves.npz",
        precision=pr,recall=rc,fpr=fpr,
        pos_hist=ph,neg_hist=nh
    )

    return {
        "Accuracy":acc,
        "Balanced_Accuracy":bal,
        "Precision":precision,
        "Recall":recall,
        "Specificity":specificity,
        "F1":f1,
        "IoU":iou,
        "MCC":mcc,
        "Kappa":kappa,
        "PR_AUC":pr_auc,
        "ROC_AUC":roc_auc,
        "Boundary_F1":float(np.mean(bf)) if bf else None,
        "TP":TP,"TN":TN,"FP":FP,"FN":FN,
        "test_inference_seconds":sec,
        "patches_per_second":len(patch_rows)/(sec+1e-12)
    }, pd.DataFrame(patch_rows)

def main():
    # JUPYTER VERSION: train exactly ONE selected experiment + seed.
    class Args:
        pass

    args = Args()
    args.data_root = r"/home/auhspark2/dhiraj/landslide"
    args.output = str(Path(args.data_root) / "Paper1_QAOA_Split")
    args.experiment = EXPERIMENT
    args.seed = int(SEED)
    args.batch_size = int(BATCH_SIZE)
    args.base_filters = int(BASE_FILTERS)
    args.epochs = int(MAX_EPOCHS)
    args.patience = int(PATIENCE)
    args.grad_accum = int(GRAD_ACCUM_STEPS)

    data_root=Path(args.data_root)
    out=Path(args.output)
    metrics_path=out/"04_metrics"/f"metrics_{args.experiment}_seed{args.seed}.json"
    patch_path=out/"04_metrics"/f"patch_counts_{args.experiment}_seed{args.seed}.csv"
    history_path=out/"04_metrics"/f"history_{args.experiment}_seed{args.seed}.csv"
    model_path=out/"03_models"/f"{args.experiment}_seed{args.seed}.pt"
    pred_prefix=out/"06_predictions"/f"{args.experiment}_seed{args.seed}"

    if metrics_path.exists() and patch_path.exists() and model_path.exists():
        print("Already complete, skipping:", args.experiment, args.seed)
        return

    if not torch.cuda.is_available():
        raise RuntimeError("PyTorch cannot see the NVIDIA GPU.")

    device=torch.device("cuda:0")
    print("="*80)
    print("STAGE 2 — ONE GPU TRAINING PROCESS")
    print("Experiment:",args.experiment,"Seed:",args.seed)
    print("GPU:",torch.cuda.get_device_name(0))
    print("Batch:",args.batch_size,"Base filters:",args.base_filters)
    print("="*80)

    set_seed(args.seed)
    torch.backends.cudnn.benchmark=False
    try:
        torch.backends.cuda.matmul.allow_tf32=True
        torch.backends.cudnn.allow_tf32=True
    except Exception:
        pass

    feature_payload=json.loads(
        (out/"02_feature_selection"/"feature_sets.json").read_text()
    )
    if args.experiment not in feature_payload["feature_sets"]:
        raise KeyError(f"Unknown experiment {args.experiment}")
    channels=feature_payload["feature_sets"][args.experiment]
    channel_names=feature_payload["channel_names"]

    norm=json.loads((out/"01_dataset"/"normalization.json").read_text())
    mean=np.asarray(norm["mean"],np.float32)
    std=np.asarray(norm["std"],np.float32)

    tr=pd.read_csv(out/"01_dataset"/"train_records.csv")
    va=pd.read_csv(out/"01_dataset"/"validation_records.csv")
    te=pd.read_csv(out/"01_dataset"/"holdout_records.csv")

    train_ds=LandslideDataset(tr,channels,mean,std,True)
    val_ds=LandslideDataset(va,channels,mean,std,False)
    test_ds=LandslideDataset(te,channels,mean,std,False)

    # num_workers=0 and pin_memory=False are deliberate for maximum stability.
    train_loader=DataLoader(
        train_ds,batch_size=args.batch_size,shuffle=True,
        num_workers=0,pin_memory=False,drop_last=False
    )
    val_loader=DataLoader(
        val_ds,batch_size=args.batch_size,shuffle=False,
        num_workers=0,pin_memory=False
    )
    test_loader=DataLoader(
        test_ds,batch_size=args.batch_size,shuffle=False,
        num_workers=0,pin_memory=False
    )

    model=ResUNet(len(channels),args.base_filters).to(device)
    params=sum(p.numel() for p in model.parameters() if p.requires_grad)
    criterion=FocalDiceLoss()
    optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,mode="min",factor=0.5,patience=3
    )
    scaler=scaler_obj()

    best=np.inf; best_epoch=-1; no_improve=0
    history=[]
    start_train=time.perf_counter()

    for epoch in range(1,args.epochs+1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total=0.0; nseen=0

        for bi,(x,y,_) in enumerate(train_loader):
            x=x.to(device,non_blocking=False)
            y=y.to(device,non_blocking=False)

            with autocast_ctx():
                logits=model(x)
                loss=criterion(logits,y)
                backward_loss=loss/args.grad_accum

            scaler.scale(backward_loss).backward()

            update=((bi+1)%args.grad_accum==0) or (bi+1==len(train_loader))
            if update:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            total += float(loss.detach().cpu())*x.size(0)
            nseen += x.size(0)
            del x,y,logits,loss,backward_loss

        train_loss=total/max(nseen,1)

        # Validation loss.
        model.eval()
        vtotal=0.0; vn=0
        with torch.no_grad():
            for x,y,_ in val_loader:
                x=x.to(device,non_blocking=False)
                y=y.to(device,non_blocking=False)
                with autocast_ctx():
                    logits=model(x)
                    vl=criterion(logits,y)
                vtotal += float(vl.detach().cpu())*x.size(0)
                vn += x.size(0)
                del x,y,logits,vl
        val_loss=vtotal/max(vn,1)
        scheduler.step(val_loss)
        lr=optimizer.param_groups[0]["lr"]
        history.append({
            "epoch":epoch,"train_loss":train_loss,
            "val_loss":val_loss,"learning_rate":lr
        })
        print(
            f"{args.experiment} seed={args.seed} "
            f"epoch={epoch:02d} train={train_loss:.5f} "
            f"val={val_loss:.5f} lr={lr:.2e}"
        )

        if val_loss < best-1e-5:
            best=val_loss; best_epoch=epoch; no_improve=0
            torch.save({
                "model_state":model.state_dict(),
                "channels":channels,
                "channel_names":[channel_names[i] for i in channels],
                "seed":args.seed,
                "experiment":args.experiment,
                "base_filters":args.base_filters,
                "best_epoch":best_epoch,
                "best_val_loss":best,
                "normalization_mean":mean.tolist(),
                "normalization_std":std.tolist(),
            },model_path)
        else:
            no_improve += 1
            if no_improve >= args.patience:
                print("Early stopping.")
                break

    train_seconds=time.perf_counter()-start_train
    pd.DataFrame(history).to_csv(history_path,index=False)

    ckpt=torch.load(model_path,map_location=device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    # Validation chooses threshold.
    vph,vnh=validation_hist(model,val_loader,device,4096)
    threshold,val_f1=best_threshold_from_hist(vph,vnh)
    print("Selected validation threshold:",threshold,"Validation F1:",val_f1)

    # Keep large per-pixel probability arrays out of Jupyter RAM.
    save_arrays=False
    metrics,patch_counts=test_metrics(
        model,test_loader,device,threshold,pred_prefix,
        bins=4096,save_arrays=save_arrays
    )
    metrics.update({
        "experiment":args.experiment,
        "seed":args.seed,
        "n_channels":len(channels),
        "channels":", ".join(channel_names[i] for i in channels),
        "threshold":threshold,
        "validation_F1_at_threshold":val_f1,
        "train_seconds":train_seconds,
        "best_epoch":best_epoch,
        "model_parameters":params,
        "batch_size":args.batch_size,
        "base_filters":args.base_filters,
        "grad_accum":args.grad_accum,
        "gpu_name":torch.cuda.get_device_name(0),
    })
    metrics_path.write_text(json.dumps(metrics,indent=2))
    patch_counts.to_csv(patch_path,index=False)

    print("\nFINAL HOLDOUT METRICS")
    for key in ["F1","IoU","Precision","Recall","MCC","PR_AUC","ROC_AUC","Boundary_F1"]:
        print(f"{key:18s}: {metrics[key]}")

    # Explicit cleanup, though process exit releases everything anyway.
    model.to("cpu")
    del model,optimizer,scaler,train_loader,val_loader,test_loader
    gc.collect()
    torch.cuda.empty_cache()
    print("\nProcess complete. Exiting now; OS will release all GPU memory.")



## Run exactly one experiment

    The cell below runs only the selected `EXPERIMENT` and `SEED`.

In [ ]:
# ============================================================
# GPU PREFLIGHT + ONE-MODEL RUN
# ============================================================

print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot see the NVIDIA GPU.")

print("GPU:", torch.cuda.get_device_name(0))
print("Starting:", EXPERIMENT, "seed", SEED)

main()

print("\n" + "=" * 80)
print("ONE EXPERIMENT FINISHED AND SAVED")
print("=" * 80)
print("Experiment:", EXPERIMENT)
print("Seed      :", SEED)
print("\nRECOMMENDED NEXT STEP:")
print("1. Restart the Jupyter kernel.")
print("2. Change EXPERIMENT and/or SEED in Step 1.")
print("3. Run this notebook again.")

## Recommended one-by-one order

For the first test, run only **ALL_14 / seed 42**.

If it finishes successfully, continue:

1. ALL_14 — 42
2. MI_8 — 42
3. mRMR_8 — 42
4. QAOA_6 — 42
5. QAOA_8 — 42
6. QAOA_10 — 42

Then repeat the same six experiments for seeds **52** and **62**.

Do **not** run the models in parallel.